In [4]:
# If needed (run once)
# !pip install pandas numpy scikit-learn matplotlib seaborn nltk

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import nltk
import re

In [5]:
df = pd.read_csv("margaret-and-david-reviews.csv")

df.head()

,user_score,genre1,genre2,studio,title,url,titles_match,year,rtId,indexNo,...,lead_actor2,lead_actor1,title-orig,davScore,margScore,dav-marg,combined-score,reviewed-at,url.1,ratio
0,93,Drama,Romance,na,Fireworks (Hana-bi),http://www.rottentomatoes.com/m/1091981-firewo...,no,1997,19686.0,2366,...,Kayoko Kishimoto,Takeshi Kitano,Hana-Bi,5.0,5.0,0.0,10.0,SBS,NaN,58.0
1,76,Drama,Romance,Indiepix,Samson and Delilah,http://www.rottentomatoes.com/m/10011295-samso...,no,2010,770808521.0,57,...,Marissa Gibson,Rowan McNamara,Samson & Delilah,5.0,5.0,0.0,10.0,ABC,http://www.abc.net.au/atthemovies/txt/s2542612...,88.0
2,83,Drama,NaN,Warner Independent Pictures,"Good Night, And Good Luck",http://www.rottentomatoes.com/m/1152019-good_n...,no,2005,8572.0,55,...,Patricia Clarkson,David Strathairn,"Good Night, and Good Luck",5.0,5.0,0.0,10.0,ABC,http://www.abc.net.au/atthemovies/txt/s1532953...,100.0
3,86,Drama,Action & Adventure,Miramax Films,No Country for Old Men,http://www.rottentomatoes.com/m/no_country_for...,no,2007,770669872.0,56,...,Javier Bardem,Tommy Lee Jones,No Country For Old Men,5.0,5.0,0.0,10.0,ABC,http://www.abc.net.au/atthemovies/txt/s2099568...,100.0
4,100,Documentary,Musical & Performing Arts,na,Facing the Music,http://www.rottentomatoes.com/m/facing_the_music/,no,2001,770753087.0,2284,...,The Renaissance Players,Anne Boyd,Facing The Music,5.0,5.0,0.0,10.0,SBS,NaN,100.0


In [6]:
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 2743 entries, 0 to 2742
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   user_score      2743 non-null   str    
 1   genre1          2743 non-null   str    
 2   genre2          2118 non-null   str    
 3   studio          2743 non-null   str    
 4   title           2743 non-null   str    
 5   url             2743 non-null   str    
 6   titles_match    2743 non-null   str    
 7   year            2743 non-null   str    
 8   rtId            2742 non-null   float64
 9   indexNo         2743 non-null   int64  
 10  director        2743 non-null   str    
 11  old_title       2743 non-null   str    
 12  mpaa_rating     2743 non-null   str    
 13  critic_score    2743 non-null   str    
 14  lead_actor2     2743 non-null   str    
 15  lead_actor1     2743 non-null   str    
 16  title-orig      2743 non-null   str    
 17  davScore        2743 non-null   float64
 18 

user_score          0
genre1              0
genre2            625
studio              0
title               0
url                 0
titles_match        0
year                0
rtId                1
indexNo             0
director            0
old_title           0
mpaa_rating         0
critic_score        0
lead_actor2         0
lead_actor1         0
title-orig          0
davScore            0
margScore           0
dav-marg            0
combined-score      0
reviewed-at         0
url.1             972
ratio              53
dtype: int64

In [9]:
df.head()

,user_score,genre1,genre2,studio,title,url,titles_match,year,rtId,indexNo,...,lead_actor2,lead_actor1,title-orig,davScore,margScore,dav-marg,combined-score,reviewed-at,url.1,ratio
0,93,Drama,Romance,na,Fireworks (Hana-bi),http://www.rottentomatoes.com/m/1091981-firewo...,no,1997,19686.0,2366,...,Kayoko Kishimoto,Takeshi Kitano,Hana-Bi,5.0,5.0,0.0,10.0,SBS,NaN,58.0
1,76,Drama,Romance,Indiepix,Samson and Delilah,http://www.rottentomatoes.com/m/10011295-samso...,no,2010,770808521.0,57,...,Marissa Gibson,Rowan McNamara,Samson & Delilah,5.0,5.0,0.0,10.0,ABC,http://www.abc.net.au/atthemovies/txt/s2542612...,88.0
2,83,Drama,NaN,Warner Independent Pictures,"Good Night, And Good Luck",http://www.rottentomatoes.com/m/1152019-good_n...,no,2005,8572.0,55,...,Patricia Clarkson,David Strathairn,"Good Night, and Good Luck",5.0,5.0,0.0,10.0,ABC,http://www.abc.net.au/atthemovies/txt/s1532953...,100.0
3,86,Drama,Action & Adventure,Miramax Films,No Country for Old Men,http://www.rottentomatoes.com/m/no_country_for...,no,2007,770669872.0,56,...,Javier Bardem,Tommy Lee Jones,No Country For Old Men,5.0,5.0,0.0,10.0,ABC,http://www.abc.net.au/atthemovies/txt/s2099568...,100.0
4,100,Documentary,Musical & Performing Arts,na,Facing the Music,http://www.rottentomatoes.com/m/facing_the_music/,no,2001,770753087.0,2284,...,The Renaissance Players,Anne Boyd,Facing The Music,5.0,5.0,0.0,10.0,SBS,NaN,100.0


In [12]:
if text_column in df.columns:
    df["clean_text"] = df[text_column].apply(clean_text)
else:
    print("Column not found. Available columns:", df.columns)

Column not found. Available columns: Index(['user_score', 'genre1', 'genre2', 'studio', 'title', 'url',
       'titles_match', 'year', 'rtId', 'indexNo', 'director', 'old_title',
       'mpaa_rating', 'critic_score', 'lead_actor2', 'lead_actor1',
       'title-orig', 'davScore', 'margScore', 'dav-marg', 'combined-score',
       'reviewed-at', 'url.1', 'ratio'],
      dtype='str')


In [16]:
df["combined_text"] = (
    df["title"].astype(str) + " " +
    df["genre1"].astype(str) + " " +
    df["genre2"].astype(str) + " " +
    df["director"].astype(str) + " " +
    df["lead_actor1"].astype(str) + " " +
    df["lead_actor2"].astype(str)
)

In [21]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X = df.select_dtypes(include='number')

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

X_clean = pipeline.fit_transform(X)

In [23]:


df.head()

,user_score,genre1,genre2,studio,title,url,titles_match,year,rtId,indexNo,...,lead_actor1,title-orig,davScore,margScore,dav-marg,combined-score,reviewed-at,url.1,ratio,combined_text
0,93,Drama,Romance,na,Fireworks (Hana-bi),http://www.rottentomatoes.com/m/1091981-firewo...,no,1997,19686.0,2366,...,Takeshi Kitano,Hana-Bi,5.0,5.0,0.0,10.0,SBS,NaN,58.0,Fireworks (Hana-bi) Drama Romance Takeshi Kita...
1,76,Drama,Romance,Indiepix,Samson and Delilah,http://www.rottentomatoes.com/m/10011295-samso...,no,2010,770808521.0,57,...,Rowan McNamara,Samson & Delilah,5.0,5.0,0.0,10.0,ABC,http://www.abc.net.au/atthemovies/txt/s2542612...,88.0,Samson and Delilah Drama Romance Warwick Thorn...
2,83,Drama,NaN,Warner Independent Pictures,"Good Night, And Good Luck",http://www.rottentomatoes.com/m/1152019-good_n...,no,2005,8572.0,55,...,David Strathairn,"Good Night, and Good Luck",5.0,5.0,0.0,10.0,ABC,http://www.abc.net.au/atthemovies/txt/s1532953...,100.0,NaN
3,86,Drama,Action & Adventure,Miramax Films,No Country for Old Men,http://www.rottentomatoes.com/m/no_country_for...,no,2007,770669872.0,56,...,Tommy Lee Jones,No Country For Old Men,5.0,5.0,0.0,10.0,ABC,http://www.abc.net.au/atthemovies/txt/s2099568...,100.0,No Country for Old Men Drama Action & Adventur...
4,100,Documentary,Musical & Performing Arts,na,Facing the Music,http://www.rottentomatoes.com/m/facing_the_music/,no,2001,770753087.0,2284,...,Anne Boyd,Facing The Music,5.0,5.0,0.0,10.0,SBS,NaN,100.0,Facing the Music Documentary Musical & Perform...


In [25]:
print(df.head())
print(df.columns)

  user_score       genre1                     genre2  \
0         93        Drama                    Romance   
1         76        Drama                    Romance   
2         83        Drama                        NaN   
3         86        Drama         Action & Adventure   
4        100  Documentary  Musical & Performing Arts   

                        studio                      title  \
0                           na        Fireworks (Hana-bi)   
1                     Indiepix         Samson and Delilah   
2  Warner Independent Pictures  Good Night, And Good Luck   
3                Miramax Films     No Country for Old Men   
4                           na           Facing the Music   

                                                 url titles_match  year  \
0  http://www.rottentomatoes.com/m/1091981-firewo...           no  1997   
1  http://www.rottentomatoes.com/m/10011295-samso...           no  2010   
2  http://www.rottentomatoes.com/m/1152019-good_n...           no  2005

In [28]:
print(type(X))

<class 'pandas.DataFrame'>


In [31]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import KMeans

texts = [
    "I love machine learning",
    "KMeans clustering is useful",
    "Text mining and NLP are fun",
    "Deep learning and AI applications",
    "I enjoy data science and AI",
    "Clustering groups similar documents"
]

k = 2  # make sure k is defined too

vectorizer = CountVectorizer(stop_words="english")
X = vectorizer.fit_transform(texts)

kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans.fit(X)

terms = vectorizer.get_feature_names_out()

for i in range(k):
    print(f"\n📌 Cluster {i} Top Words:")

    center = kmeans.cluster_centers_[i]
    top_indices = center.argsort()[-10:][::-1]

    for idx in top_indices:
        print(terms[idx], end=" ")
    print("\n")


📌 Cluster 0 Top Words:
ai learning enjoy applications science data machine love deep clustering 


📌 Cluster 1 Top Words:
clustering useful text documents fun groups kmeans mining nlp similar 

